In [ ]:
!git clone https://github.com/Stanford-AIMI/CheXagent.git


In [ ]:
import os
import sys

In [ ]:
# @title Authenticate with HuggingFace, skip if you have a HF_TOKEN secret

# Authenticate user for HuggingFace if needed. Enter token below if requested.
from huggingface_hub.utils import HfFolder
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

In [ ]:
# ---- Pin to CheXagent-compatible stack ----
%pip -q install -U pip setuptools wheel

# 1) Remove conflicting packages (ignore "not installed" warnings)
%pip -q uninstall -y transformers==4.41.0 peft accelerate bitsandbytes \
  trl tensorflow tensorflow-decision-forests numba \
  cuml-cu12 cudf-cu12 dask-cuda distributed-ucxx-cu12 umap-learn librosa shap \
  stumpy pynndescent dopamine-rl spacy thinc opencv-python opencv-contrib-python opencv-python-headless \
  albumentations albucore || true

%pip -q uninstall -y sentence-transformers tsfresh stumpy \
  dask-cudf-cu12 cudf-cu12 cuml-cu12 dask-cuda distributed-ucxx-cu12 || true

# 2) Install versions that work with CheXagent's remote code
%pip -q install "numpy==2.0.2" pillow
%pip -q install "transformers==4.40.0" "peft==0.11.1" "accelerate==0.29.3" "bitsandbytes==0.43.1"
# CheXagent imports these at import time:
%pip -q install "opencv-python-headless==4.12.0.88" "albumentations==2.0.8" "albucore==0.0.24"

# (Often needed by vision stacks; cheap to include)
%pip -q install einops timm sentencepiece

import numpy, transformers, peft, cv2, albumentations as A, timm, einops
print("NumPy", numpy.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| cv2", cv2.__version__, "| albumentations", A.__version__)
print("Go to Runtime > Restart runtime, then re-run your CheXagent load + fine_tune call.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from PIL import Image

class CheXagent(object):
    def __init__(self):
        # step 1: Setup constant
        self.model_name = "StanfordAIMI/CheXagent-2-3b"
        self.dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.device = "cuda"

        # step 2: Load Processor and Model
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name, device_map="auto", torch_dtype=self.dtype, trust_remote_code=True)
        #self.model = self.model.to(self.dtype)
        self.model.eval()

    def generate(self, paths, prompt):
        # step 3: Inference
        query = self.tokenizer.from_list_format([*[{'image': path} for path in paths], {'text': prompt}])
        conv = [{"from": "system", "value": "You are a helpful assistant."}, {"from": "human", "value": query}]
        input_ids = self.tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
        output = self.model.generate(
            input_ids.to(self.device), do_sample=False, num_beams=1, temperature=1., top_p=1., use_cache=True,
            max_new_tokens=512
        )[0]
        response = self.tokenizer.decode(output[input_ids.size(1):-1])
        return response

    def view_classification(self, path):
        assert isinstance(path, str)
        prompt = "What is the view of this chest X-ray? Options: (a) PA, (b) AP, (c) LATERAL"
        response = self.generate([path], prompt)
        return response


In [ ]:
from rich import print

def main():
    # Load the model
    chexagent = CheXagent()

    # Task 1: View Classification
    path = "https://prod-images-static.radiopaedia.org/images/23511538/8a28003cc78f3549ac9f436dfe7dad_big_gallery.jpeg"
    response = chexagent.view_classification(path)
    print(f'=' * 42)
    print(f'[Task 1: View Classification]')
    print(f'Image: {path}')
    print(f'Result: {response}')
    print(f'=' * 42)



if __name__ == '__main__':
    main()